# FDKD — Fusion Decoupled Knowledge Distillation
## Google Colab: Miniforge + Conda + Export + Server

Chạy từng cell từ trên xuống dưới.

In [ ]:
# ── Cell 1: Cài Miniforge + tạo môi trường Conda Python 3.10 ──
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash Miniforge3-Linux-x86_64.sh -b -p /opt/conda 2>/dev/null
import os
os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

!conda remove -n openmmlab --all -y -q 2>/dev/null
!conda create --name openmmlab python=3.10 -y -q

print("✅ Miniforge + conda env 'openmmlab' ready")

In [ ]:
# ── Cell 2: Clone repo + mount Drive ──
%cd /content
import os
if not os.path.exists('/content/FDKD-Fusion-Decoupled-Knowledge-Distillation'):
    !git clone https://github.com/MingNhayMua/FDKD-Fusion-Decoupled-Knowledge-Distillation.git
%cd /content/FDKD-Fusion-Decoupled-Knowledge-Distillation

from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = "/content/drive/MyDrive/CS338-checkpoint"
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"Exists: {os.path.isdir(CKPT_DIR)}")
if os.path.isdir(CKPT_DIR):
    !ls "$CKPT_DIR"

In [ ]:
# ── Cell 3: Cài PyTorch + OpenMMLab + backend deps ──
!conda run -n openmmlab pip install torch==2.1.0 torchvision==0.16.0 "numpy<2" --index-url https://download.pytorch.org/whl/cu121 -q
!conda run -n openmmlab pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html "numpy<2" -q
!conda run -n openmmlab pip install -U openmim "numpy<2" -q
!conda run -n openmmlab mim install "mmpretrain>=1.0.0" -q
!conda run -n openmmlab pip install -r backend/requirements.txt "numpy<2" -q

print("✅ All dependencies installed")

In [ ]:
# ── Cell 4: Export models to TorchScript (skip nếu đã có traced) ──
!conda run -n openmmlab python export_models.py --checkpoint-dir "$CKPT_DIR"

In [ ]:
# ── Cell 5: Start server ──
NGROK_TOKEN = ""  # ← điền ngrok auth token

!conda run --no-capture-output -n openmmlab python run_colab.py --checkpoint-dir "$CKPT_DIR" --token "$NGROK_TOKEN" --port 8000

## Sau khi chạy xong

- Copy **public URL** (dạng `https://xxxx.ngrok-free.app`) từ output Cell 5
- Paste vào frontend connection input

## Lưu ý
- Cell 4 có thể skip nếu `*_traced.pt` đã tồn tại trong `CKPT_DIR`
- Điền `NGROK_TOKEN` ở Cell 5 để tránh giới hạn 2h free tier